# Automated Card Feeder Integration

This notebook implements high-speed card processing for an automated feeder system.
Target: Process 1 card every 2 seconds using video stream processing.

## Key Optimizations:
- Video stream processing instead of timed photos
- GPU acceleration for neural network inference
- Asynchronous processing pipeline
- Motion detection for card positioning
- Performance monitoring and error handling

In [ ]:
import cv2
import numpy as np
import tensorflow as tf
import threading
import time
import asyncio
import concurrent.futures
import logging
from queue import Queue
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
import pinecone
from pinecone.grpc import PineconeGRPC as Pinecone
import os
import requests
import csv

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

## GPU Optimization Setup

In [ ]:
# Configure GPU for optimal performance
physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    print(f"GPU detected: {physical_devices[0]}")
    tf.config.experimental.set_memory_growth(physical_devices[0], True)
    # Enable mixed precision for faster inference
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
else:
    print("No GPU detected. Using CPU.")
    # Optimize CPU usage
    tf.config.threading.set_intra_op_parallelism_threads(8)
    tf.config.threading.set_inter_op_parallelism_threads(8)

## Load and Optimize Model

In [ ]:
# Load EfficientNetB0 model with optimizations
print("Loading EfficientNetB0 model...")
base_model = EfficientNetB0(weights='imagenet', include_top=False, pooling='avg')

# Warm up the model with a dummy prediction
dummy_input = np.random.random((1, 224, 224, 3)).astype(np.float32)
_ = base_model.predict(dummy_input, verbose=0)
print("Model loaded and warmed up.")

# Initialize Pinecone
api_key = os.getenv("PINECONE_API_KEY")
if not api_key:
    print("Warning: PINECONE_API_KEY not found in environment variables")
else:
    pinecone_client = pinecone.Pinecone(api_key=api_key)
    index = pinecone_client.Index('mtg-cards-index-efficientnet')
    print("Pinecone client initialized.")

## Performance Monitoring Class

In [ ]:
class PerformanceMonitor:
    def __init__(self):
        self.processing_times = []
        self.card_count = 0
        self.start_time = time.time()
        self.errors = 0
        
    def log_processing_time(self, processing_time):
        self.processing_times.append(processing_time)
        self.card_count += 1
        
        if processing_time > 2.0:
            logging.warning(f"Slow processing detected: {processing_time:.2f}s")
    
    def log_error(self):
        self.errors += 1
        
    def get_stats(self):
        if not self.processing_times:
            return {}
            
        avg_time = sum(self.processing_times) / len(self.processing_times)
        max_time = max(self.processing_times)
        min_time = min(self.processing_times)
        total_runtime = time.time() - self.start_time
        throughput = self.card_count / (total_runtime / 60) if total_runtime > 0 else 0
        
        return {
            'cards_processed': self.card_count,
            'avg_processing_time': avg_time,
            'max_processing_time': max_time,
            'min_processing_time': min_time,
            'cards_per_minute': throughput,
            'total_runtime': total_runtime,
            'errors': self.errors,
            'success_rate': (self.card_count / (self.card_count + self.errors)) * 100 if self.card_count + self.errors > 0 else 0
        }
    
    def print_stats(self):
        stats = self.get_stats()
        if stats:
            print(f"\n=== Performance Statistics ===")
            print(f"Cards Processed: {stats['cards_processed']}")
            print(f"Average Processing Time: {stats['avg_processing_time']:.3f}s")
            print(f"Max Processing Time: {stats['max_processing_time']:.3f}s")
            print(f"Min Processing Time: {stats['min_processing_time']:.3f}s")
            print(f"Throughput: {stats['cards_per_minute']:.1f} cards/minute")
            print(f"Total Runtime: {stats['total_runtime']:.1f}s")
            print(f"Errors: {stats['errors']}")
            print(f"Success Rate: {stats['success_rate']:.1f}%")

# Global performance monitor
perf_monitor = PerformanceMonitor()

## Optimized Image Processing Functions

In [ ]:
def resize_with_padding_fast(img, target_size=224):
    """Optimized version of resize with padding using OpenCV"""
    h, w = img.shape[:2]
    scale = min(target_size / h, target_size / w)
    new_w = int(w * scale)
    new_h = int(h * scale)
    
    # Use INTER_LINEAR for faster resizing (vs INTER_CUBIC)
    resized_img = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    
    # Create padded image
    padded_img = np.full((target_size, target_size, 3), 255, dtype=np.uint8)
    pad_w = (target_size - new_w) // 2
    pad_h = (target_size - new_h) // 2
    padded_img[pad_h:pad_h + new_h, pad_w:pad_w + new_w, :] = resized_img
    
    return padded_img

def preprocess_frame_fast(frame):
    """Fast preprocessing for video frames"""
    # Resize with padding
    processed = resize_with_padding_fast(frame, 224)
    
    # Convert to float32 and normalize
    processed = processed.astype(np.float32)
    processed = np.expand_dims(processed, axis=0)
    processed = preprocess_input(processed)
    
    return processed

def embed_image_fast(frame):
    """Fast embedding generation from frame"""
    try:
        processed_frame = preprocess_frame_fast(frame)
        embedding = base_model.predict(processed_frame, verbose=0)
        return embedding.flatten()
    except Exception as e:
        logging.error(f"Error in embed_image_fast: {e}")
        return None

## Motion Detection for Card Positioning

In [ ]:
class CardMotionDetector:
    def __init__(self, threshold=1000, min_card_area=50000):
        self.background_subtractor = cv2.createBackgroundSubtractorMOG2(
            detectShadows=False, varThreshold=50
        )
        self.threshold = threshold
        self.min_card_area = min_card_area
        self.card_stable_frames = 0
        self.required_stable_frames = 5  # Card must be stable for 5 frames
        
    def detect_card_present(self, frame):
        """Detect if a card is present and stable in the frame"""
        # Apply background subtraction
        fg_mask = self.background_subtractor.apply(frame)
        
        # Find contours
        contours, _ = cv2.findContours(fg_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        # Check for card-sized objects
        for contour in contours:
            area = cv2.contourArea(contour)
            if area > self.min_card_area:
                # Check if contour is roughly rectangular (card-like)
                peri = cv2.arcLength(contour, True)
                approx = cv2.approxPolyDP(contour, 0.02 * peri, True)
                
                if len(approx) >= 4:  # Roughly rectangular
                    self.card_stable_frames += 1
                    if self.card_stable_frames >= self.required_stable_frames:
                        return True
                    return False
        
        # No card detected, reset counter
        self.card_stable_frames = 0
        return False
    
    def reset_detection(self):
        """Reset the motion detector (call after processing a card)"""
        self.card_stable_frames = 0

motion_detector = CardMotionDetector()

## Database Query and Catalog Management

In [ ]:
def query_card_database(embedding, top_k=3):
    """Query the Pinecone database for similar cards"""
    try:
        query_results = index.query(
            namespace="mtg_cards",
            vector=embedding.tolist(),
            top_k=top_k
        )
        return query_results
    except Exception as e:
        logging.error(f"Database query error: {e}")
        return None

def get_card_details_fast(card_id):
    """Fast card details retrieval from Scryfall API"""
    try:
        scryfall_api_url = f"https://api.scryfall.com/cards/{card_id}"
        response = requests.get(scryfall_api_url, timeout=5)
        if response.status_code == 200:
            return response.json()
        else:
            logging.error(f"Scryfall API error: {response.status_code}")
            return None
    except Exception as e:
        logging.error(f"Error fetching card details: {e}")
        return None

def update_catalog_fast(card_data, file_path='Output/mtg_catalog.csv'):
    """Fast catalog update using in-memory operations"""
    try:
        # Read existing data if file exists
        if os.path.isfile(file_path):
            with open(file_path, mode='r', newline='', encoding='utf-8') as file:
                reader = csv.DictReader(file)
                rows = list(reader)
        else:
            rows = []
        
        # Extract relevant data
        card_id = card_data['id']
        name = card_data['name']
        price = float(card_data.get('prices', {}).get('usd', 0) or 0)
        
        # Check if card already exists
        card_found = False
        for row in rows:
            if row['id'] == card_id:
                row['number_owned'] = int(row['number_owned']) + 1
                row['total_value'] = int(row['number_owned']) * price
                card_found = True
                break
        
        if not card_found:
            # Add new card
            new_row = {
                "id": card_id,
                "name": name,
                "mana_cost": card_data.get('mana_cost', ''),
                "cmc": card_data.get('cmc', 0),
                "type_line": card_data.get('type_line', '').replace('—', '-'),
                "colors": card_data.get('colors', []),
                "color_identity": card_data.get('color_identity', []),
                "set_name": card_data.get('set_name', ''),
                "rarity": card_data.get('rarity', ''),
                "full_art": card_data.get('full_art', False),
                "price": price,
                "number_owned": 1,
                "total_value": price
            }
            rows.append(new_row)
        
        # Write updated data back to file
        fieldnames = ["id", "name", "mana_cost", "cmc", "type_line", "colors", 
                     "color_identity", "set_name", "rarity", "full_art", "price", 
                     "number_owned", "total_value"]
        
        with open(file_path, mode='w', newline='', encoding='utf-8') as file:
            writer = csv.DictWriter(file, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(rows)
        
        return True
        
    except Exception as e:
        logging.error(f"Error updating catalog: {e}")
        return False

## Main Card Processing Pipeline

In [ ]:
class HighSpeedCardProcessor:
    def __init__(self, camera_index=0, confidence_threshold=0.85):
        self.camera_index = camera_index
        self.confidence_threshold = confidence_threshold
        self.video_capture = None
        self.processing = False
        self.frame_queue = Queue(maxsize=30)  # Buffer frames
        self.last_processed_time = 0
        self.min_processing_interval = 2.0  # Minimum 2 seconds between cards
        
    def initialize_camera(self):
        """Initialize camera with optimal settings"""
        self.video_capture = cv2.VideoCapture(self.camera_index)
        
        # Set camera properties for optimal performance
        self.video_capture.set(cv2.CAP_PROP_FRAME_WIDTH, 1920)
        self.video_capture.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)
        self.video_capture.set(cv2.CAP_PROP_FPS, 30)
        self.video_capture.set(cv2.CAP_PROP_BUFFERSIZE, 1)  # Minimize latency
        
        if not self.video_capture.isOpened():
            raise Exception(f"Could not open camera {self.camera_index}")
        
        logging.info(f"Camera initialized: {self.camera_index}")
    
    def process_single_card(self, frame):
        """Process a single card from frame"""
        start_time = time.time()
        
        try:
            # Generate embedding
            embedding = embed_image_fast(frame)
            if embedding is None:
                return None
            
            # Query database
            query_results = query_card_database(embedding)
            if not query_results or not query_results.matches:
                logging.warning("No matches found in database")
                return None
            
            # Get best match
            best_match = query_results.matches[0]
            confidence = best_match.score
            
            if confidence < self.confidence_threshold:
                logging.warning(f"Low confidence match: {confidence:.3f}")
                return None
            
            # Extract card ID and get details
            card_id = best_match.id.split('.')[0]  # Remove file extension
            card_data = get_card_details_fast(card_id)
            
            if card_data:
                # Update catalog
                success = update_catalog_fast(card_data)
                
                processing_time = time.time() - start_time
                perf_monitor.log_processing_time(processing_time)
                
                result = {
                    'name': card_data['name'],
                    'set': card_data.get('set_name', 'Unknown'),
                    'price': card_data.get('prices', {}).get('usd', 'N/A'),
                    'confidence': confidence,
                    'processing_time': processing_time,
                    'catalog_updated': success
                }
                
                logging.info(f"Card processed: {result['name']} ({processing_time:.3f}s)")
                return result
            
        except Exception as e:
            logging.error(f"Error processing card: {e}")
            perf_monitor.log_error()
        
        return None
    
    def capture_and_process(self, duration_seconds=60, display_video=False):
        """Main processing loop"""
        if not self.video_capture:
            self.initialize_camera()
        
        self.processing = True
        start_time = time.time()
        
        logging.info(f"Starting card processing for {duration_seconds} seconds...")
        
        try:
            while self.processing and (time.time() - start_time) < duration_seconds:
                ret, frame = self.video_capture.read()
                if not ret:
                    logging.error("Failed to capture frame")
                    break
                
                # Check if enough time has passed since last processing
                current_time = time.time()
                if current_time - self.last_processed_time < self.min_processing_interval:
                    if display_video:
                        cv2.imshow('Card Feeder', frame)
                        if cv2.waitKey(1) & 0xFF == ord('q'):
                            break
                    continue
                
                # Detect if card is present and stable
                if motion_detector.detect_card_present(frame):
                    logging.info("Card detected, processing...")
                    
                    # Process the card
                    result = self.process_single_card(frame)
                    if result:
                        print(f"✓ {result['name']} - ${result['price']} ({result['confidence']:.3f})")
                    else:
                        print("✗ Card processing failed")
                    
                    # Reset detection and update timing
                    motion_detector.reset_detection()
                    self.last_processed_time = current_time
                
                # Display video if requested
                if display_video:
                    cv2.imshow('Card Feeder', frame)
                    if cv2.waitKey(1) & 0xFF == ord('q'):
                        break
        
        except KeyboardInterrupt:
            logging.info("Processing interrupted by user")
        
        finally:
            self.stop_processing()
            perf_monitor.print_stats()
    
    def stop_processing(self):
        """Stop processing and cleanup"""
        self.processing = False
        if self.video_capture:
            self.video_capture.release()
        cv2.destroyAllWindows()
        logging.info("Processing stopped")

# Initialize the processor
processor = HighSpeedCardProcessor(camera_index=0, confidence_threshold=0.80)

## Test Camera and Motion Detection

In [ ]:
# Test camera initialization and display feed
def test_camera_feed(duration=10):
    """Test camera feed and motion detection for specified duration"""
    processor.initialize_camera()
    
    start_time = time.time()
    print(f"Testing camera feed for {duration} seconds. Press 'q' to quit early.")
    
    while (time.time() - start_time) < duration:
        ret, frame = processor.video_capture.read()
        if not ret:
            print("Failed to capture frame")
            break
        
        # Test motion detection
        card_detected = motion_detector.detect_card_present(frame)
        
        # Add status text to frame
        status_text = "CARD DETECTED" if card_detected else "Waiting for card..."
        color = (0, 255, 0) if card_detected else (0, 255, 255)
        cv2.putText(frame, status_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)
        
        cv2.imshow('Camera Test', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    processor.stop_processing()
    print("Camera test completed.")

# Uncomment to test camera
# test_camera_feed(10)

## Run High-Speed Card Processing

In [ ]:
# Start the high-speed card processing
# This will process cards for 60 seconds or until interrupted

print("Starting high-speed card processing...")
print("Place cards in the feeder. The system will automatically detect and process them.")
print("Press Ctrl+C to stop processing early.")
print("\nProcessing results:")
print("-" * 50)

# Run for 60 seconds with video display
processor.capture_and_process(duration_seconds=60, display_video=True)

## Performance Analysis and Optimization Tips

In [ ]:
# Display detailed performance statistics
def analyze_performance():
    stats = perf_monitor.get_stats()
    
    if not stats:
        print("No performance data available.")
        return
    
    print("\n" + "="*60)
    print("PERFORMANCE ANALYSIS")
    print("="*60)
    
    print(f"📊 Cards Processed: {stats['cards_processed']}")
    print(f"⚡ Average Processing Time: {stats['avg_processing_time']:.3f}s")
    print(f"🏃 Throughput: {stats['cards_per_minute']:.1f} cards/minute")
    print(f"✅ Success Rate: {stats['success_rate']:.1f}%")
    
    # Performance recommendations
    print("\n📋 OPTIMIZATION RECOMMENDATIONS:")
    
    if stats['avg_processing_time'] > 2.0:
        print("⚠️  Average processing time exceeds 2s target")
        print("   • Consider reducing image resolution")
        print("   • Enable GPU acceleration if available")
        print("   • Reduce confidence threshold if accuracy allows")
    else:
        print("✅ Processing time within 2s target")
    
    if stats['success_rate'] < 95:
        print("⚠️  Success rate below 95%")
        print("   • Check lighting conditions")
        print("   • Verify card positioning")
        print("   • Review motion detection settings")
    else:
        print("✅ High success rate achieved")
    
    if stats['cards_per_minute'] < 30:
        print("⚠️  Throughput below optimal (30 cards/minute)")
        print("   • Reduce minimum processing interval")
        print("   • Optimize motion detection sensitivity")
        print("   • Consider batch processing techniques")
    else:
        print("✅ Excellent throughput achieved")
    
    print("\n🔧 SYSTEM OPTIMIZATION CHECKLIST:")
    gpu_available = len(tf.config.list_physical_devices('GPU')) > 0
    print(f"   GPU Acceleration: {'✅ Enabled' if gpu_available else '❌ Not Available'}")
    print(f"   Model Warm-up: ✅ Completed")
    print(f"   Motion Detection: ✅ Active")
    print(f"   Performance Monitoring: ✅ Active")

# Run performance analysis
analyze_performance()

## Batch Processing Mode (Alternative Approach)

In [ ]:
def process_image_directory_batch(directory_path, batch_size=8):
    """Process multiple images in batches for maximum throughput"""
    import os
    from PIL import Image
    
    image_files = [f for f in os.listdir(directory_path) 
                  if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    
    print(f"Processing {len(image_files)} images in batches of {batch_size}")
    
    start_time = time.time()
    
    for i in range(0, len(image_files), batch_size):
        batch_files = image_files[i:i+batch_size]
        batch_frames = []
        
        # Load batch of images
        for filename in batch_files:
            filepath = os.path.join(directory_path, filename)
            frame = cv2.imread(filepath)
            if frame is not None:
                batch_frames.append(frame)
        
        # Process batch
        if batch_frames:
            batch_start = time.time()
            
            # Preprocess all images
            processed_frames = [preprocess_frame_fast(frame) for frame in batch_frames]
            
            # Generate embeddings in batch
            batch_array = np.vstack(processed_frames)
            embeddings = base_model.predict(batch_array, verbose=0)
            
            # Process each embedding
            for j, embedding in enumerate(embeddings):
                card_start = time.time()
                
                query_results = query_card_database(embedding)
                if query_results and query_results.matches:
                    best_match = query_results.matches[0]
                    if best_match.score > 0.80:
                        card_id = best_match.id.split('.')[0]
                        card_data = get_card_details_fast(card_id)
                        if card_data:
                            update_catalog_fast(card_data)
                            card_time = time.time() - card_start
                            perf_monitor.log_processing_time(card_time)
                            print(f"✓ {card_data['name']} ({card_time:.3f}s)")
            
            batch_time = time.time() - batch_start
            print(f"Batch {i//batch_size + 1} completed in {batch_time:.3f}s")
    
    total_time = time.time() - start_time
    print(f"\nBatch processing completed in {total_time:.2f}s")
    print(f"Average: {total_time/len(image_files):.3f}s per image")
    perf_monitor.print_stats()

# Example usage:
# process_image_directory_batch('Datasets/mtg_test_images', batch_size=4)